# Raw IB Low-Level Test

Minimales Notebook — direkte `ib_async`-Calls, **kein** `IBClient`-Wrapper.
Alle Parameter stehen in der ersten Code-Zelle.  
Jeder Block kann unabhängig ausgeführt werden (nach erfolgtem Connect).

## 0 — Setup & Parameter

In [ ]:
# ── Verbindung ────────────────────────────────────────────────────────────────
HOST        = '127.0.0.1'
HOST        =  'nova-dev'
PORT        = 4001          # 4001 = Gateway Live, 4002 = Gateway Paper
                            # 7496 = TWS Live,     7497 = TWS Paper
CLIENT_ID   = 98            # andere ID als Haupt-Scanner verwenden!
MKT_DATA    = 1             # 1=live  2=frozen  3=delayed  4=delayed-frozen

# ── Stock ─────────────────────────────────────────────────────────────────────
SYMBOL      = 'AAPL'
EXCHANGE    = 'SMART'
PRIMARY_EX  = 'NASDAQ'        # für europäische Aktien wichtig
CURRENCY    = 'USD'

# ── Option (für Test 3) ───────────────────────────────────────────────────────
OPT_EXPIRY  = '20280121'    # YYYYMMDD
OPT_STRIKE  = 250.0
OPT_RIGHT   = 'P'           # 'P' = Put  |  'C' = Call
OPT_TRADING_CLASS = 'AAPL'   # aus Option-Chain-Test übernehmen (Test 2)

# ── Imports & Event-Loop ──────────────────────────────────────────────────────
from ib_async import IB, Stock, Option, util
import math, pprint

util.startLoop()
print('Setup OK')

Setup OK


## 1 — Connect

In [2]:
ib = IB()
ib.connect(HOST, PORT, clientId=CLIENT_ID, timeout=15)
ib.reqMarketDataType(MKT_DATA)

print(f'Connected : {ib.isConnected()}')
print(f'Server    : {ib.client.serverVersion()}')
print(f'MktData   : type {MKT_DATA}')

Connected : True
Server    : 178
MktData   : type 4


---
## Test 1 — Stock-Preis

Drei unabhängige Methoden, damit sichtbar ist, welche davon Daten liefert.

| Methode | Wann verfügbar |
|---|---|
| `reqMktData` snapshot | US mit Delayed-Abo oder Live-Subscription |
| `reqHistoricalData` | Funktioniert für **alle** Börsen, kein Abo nötig |
| `reqMktData` stream | Gleiche Einschränkung wie Snapshot |

In [15]:
# ── Kontrakt qualifizieren ────────────────────────────────────────────────────
stk = Stock(SYMBOL, EXCHANGE, CURRENCY)
stk.primaryExch = PRIMARY_EX
ib.qualifyContracts(stk)

print('Qualifizierter Kontrakt:')
print(f'  conId          : {stk.conId}')
print(f'  symbol         : {stk.symbol}')
print(f'  exchange       : {stk.exchange}')
print(f'  primaryExchange: {stk.primaryExchange}')
print(f'  currency       : {stk.currency}')
print(f'  localSymbol    : {stk.localSymbol}')
print(f'  tradingClass   : {stk.tradingClass}')

Qualifizierter Kontrakt:
  conId          : 265598
  symbol         : AAPL
  exchange       : SMART
  primaryExchange: NASDAQ
  currency       : USD
  localSymbol    : AAPL
  tradingClass   : NMS


In [16]:
# ── Methode A: reqMktData snapshot ────────────────────────────────────────────
ticker_snap = ib.reqMktData(stk, '', snapshot=True, regulatorySnapshot=False)
ib.sleep(2.5)

print('=== reqMktData snapshot ===')
print(f'  last      : {ticker_snap.last}')
print(f'  bid       : {ticker_snap.bid}')
print(f'  ask       : {ticker_snap.ask}')
print(f'  close     : {ticker_snap.close}')
print(f'  marketPrice: {ticker_snap.marketPrice()}')

=== reqMktData snapshot ===
  last      : 273.28
  bid       : 273.26
  ask       : 273.3
  close     : 270.23
  marketPrice: 273.28


In [17]:
# ── Methode B: reqHistoricalData (kein Abo nötig) ─────────────────────────────
bars = ib.reqHistoricalData(
    stk,
    endDateTime='',        # letzter verfügbarer Zeitpunkt
    durationStr='5 D',     # 5 Handelstage zurück (überbrückt Feiertage)
    barSizeSetting='1 day',
    whatToShow='TRADES',
    useRTH=True,
    formatDate=1,
    keepUpToDate=False,
)

print('=== reqHistoricalData (Daily Bars) ===')
if bars:
    for b in bars:
        print(f'  {b.date}  O={b.open:.2f}  H={b.high:.2f}  L={b.low:.2f}  C={b.close:.2f}  Vol={b.volume}')
    print(f'\n→ Letzter Schlusskurs: {bars[-1].close:.2f} {CURRENCY}')
else:
    print('  Keine Bars erhalten.')

=== reqHistoricalData (Daily Bars) ===
  2026-04-14  O=259.11  H=261.93  L=257.19  C=258.83  Vol=21882353.0
  2026-04-15  O=258.04  H=266.56  L=257.81  C=266.43  Vol=26573117.0
  2026-04-16  O=266.80  H=267.16  L=261.27  C=263.40  Vol=23247234.0
  2026-04-17  O=267.07  H=272.30  L=266.72  C=270.23  Vol=27891250.0
  2026-04-20  O=270.33  H=274.28  L=270.33  C=273.23  Vol=5780390.0

→ Letzter Schlusskurs: 273.23 USD


In [18]:
# ── Methode C: reqMktData streaming (30 s) ────────────────────────────────────
ticker_stream = ib.reqMktData(stk, '', snapshot=False, regulatorySnapshot=False)
ib.sleep(4.0)

print('=== reqMktData streaming (nach 4 s) ===')
print(f'  last      : {ticker_stream.last}')
print(f'  bid       : {ticker_stream.bid}')
print(f'  ask       : {ticker_stream.ask}')
print(f'  close     : {ticker_stream.close}')
print(f'  marketPrice: {ticker_stream.marketPrice()}')

ib.cancelMktData(stk)
print('→ Stream abbestellt.')

=== reqMktData streaming (nach 4 s) ===
  last      : 273.45
  bid       : 273.44
  ask       : 273.47
  close     : 270.23
  marketPrice: 273.45
→ Stream abbestellt.


---
## Test 2 — Option-Chain-Parameter

`reqSecDefOptParams` liefert alle von IB gelisteten Expiries und Strikes für das Underlying.  
Kein Market-Data-Abo notwendig.

In [19]:
# ── rohe SecDefOptParams abrufen ──────────────────────────────────────────────
params = ib.reqSecDefOptParams(
    stk.symbol,
    '',             # futFopExchange (leer = Equity)
    stk.secType,    # 'STK'
    stk.conId,
)

print(f'{len(params)} Parametersets erhalten:')  
for p in params:
    print(f'  exchange={p.exchange:8s}  tradingClass={p.tradingClass:10s}  '
          f'expiries={len(p.expirations)}  strikes={len(p.strikes)}')

39 Parametersets erhalten:
  exchange=CBOE      tradingClass=AAPL        expiries=25  strikes=117
  exchange=ISE       tradingClass=AAPL        expiries=25  strikes=117
  exchange=EMERALD   tradingClass=AAPL        expiries=25  strikes=117
  exchange=PHLX      tradingClass=AAPL        expiries=25  strikes=117
  exchange=MEMX      tradingClass=2AAPL       expiries=1  strikes=1
  exchange=EMERALD   tradingClass=2AAPL       expiries=1  strikes=1
  exchange=PEARL     tradingClass=2AAPL       expiries=1  strikes=1
  exchange=PHLX      tradingClass=2AAPL       expiries=1  strikes=1
  exchange=MEMX      tradingClass=AAPL        expiries=25  strikes=117
  exchange=PSE       tradingClass=AAPL        expiries=25  strikes=117
  exchange=EDGX      tradingClass=2AAPL       expiries=1  strikes=1
  exchange=BATS      tradingClass=AAPL        expiries=25  strikes=117
  exchange=BOX       tradingClass=2AAPL       expiries=1  strikes=1
  exchange=GEMINI    tradingClass=2AAPL       expiries=1  strikes=1


In [20]:
# ── SMART-Chain auswählen und inspizieren ─────────────────────────────────────
chain = next((p for p in params if p.exchange == 'SMART'), params[0] if params else None)

if chain is None:
    print('Keine Chain gefunden!')
else:
    expiries = sorted(chain.expirations)
    strikes  = sorted(chain.strikes)

    print(f'Exchange     : {chain.exchange}')
    print(f'TradingClass : {chain.tradingClass}')
    print(f'Expiries     : {len(expiries)} Stück')
    print(f'  erste 10   : {expiries[:10]}')
    print(f'  letzte 5   : {expiries[-5:]}')
    print(f'Strikes      : {len(strikes)} Stück')
    print(f'  Bereich    : {strikes[0]:.2f} – {strikes[-1]:.2f}')
    print(f'  Alle       : {strikes}')
    print()
    print(f'→ OPT_TRADING_CLASS für Test 3 setzen auf: "{chain.tradingClass}"')

Exchange     : SMART
TradingClass : AAPL
Expiries     : 25 Stück
  erste 10   : ['20260420', '20260422', '20260424', '20260427', '20260429', '20260501', '20260504', '20260508', '20260515', '20260522']
  letzte 5   : ['20270617', '20271217', '20280121', '20280317', '20281215']
Strikes      : 117 Stück
  Bereich    : 5.00 – 550.00
  Alle       : [5.0, 10.0, 15.0, 20.0, 25.0, 30.0, 35.0, 40.0, 45.0, 50.0, 55.0, 60.0, 65.0, 70.0, 75.0, 80.0, 85.0, 90.0, 95.0, 100.0, 105.0, 110.0, 115.0, 120.0, 125.0, 130.0, 135.0, 140.0, 145.0, 150.0, 155.0, 160.0, 165.0, 170.0, 175.0, 180.0, 185.0, 190.0, 195.0, 200.0, 205.0, 210.0, 212.5, 215.0, 217.5, 220.0, 222.5, 225.0, 227.5, 230.0, 232.5, 235.0, 237.5, 240.0, 242.5, 245.0, 247.5, 250.0, 252.5, 255.0, 257.5, 260.0, 262.5, 265.0, 267.5, 270.0, 272.5, 275.0, 277.5, 280.0, 282.5, 285.0, 287.5, 290.0, 292.5, 295.0, 297.5, 300.0, 305.0, 310.0, 315.0, 320.0, 325.0, 330.0, 335.0, 340.0, 345.0, 350.0, 355.0, 360.0, 365.0, 370.0, 375.0, 380.0, 385.0, 390.0, 3

---
## Test 3 — Quote für eine einzelne Option

Direkte Qualifizierung + Market-Data-Request für genau einen Kontrakt.  
Strike und Expiry aus den Parametern oben wählen (Zelle 0).

In [21]:
# ── Option-Kontrakt bauen + qualifizieren ─────────────────────────────────────
opt = Option(
    symbol                       = SYMBOL,
    lastTradeDateOrContractMonth = OPT_EXPIRY,
    strike                       = OPT_STRIKE,
    right                        = OPT_RIGHT,
    exchange                     = EXCHANGE,
    currency                     = CURRENCY,
    tradingClass                 = OPT_TRADING_CLASS,
    multiplier                   = '100',
)

qualified = ib.qualifyContracts(opt)

print('=== Qualifizierter Options-Kontrakt ===')
if not opt.conId:
    print('  ❌ Qualifizierung fehlgeschlagen — Strike/Expiry existiert nicht bei IB.')
    print('     → Strike oder Expiry aus Test 2 anpassen.')
else:
    print(f'  conId          : {opt.conId}')
    print(f'  symbol         : {opt.symbol}')
    print(f'  expiry         : {opt.lastTradeDateOrContractMonth}')
    print(f'  strike         : {opt.strike}')
    print(f'  right          : {opt.right}')
    print(f'  exchange       : {opt.exchange}')
    print(f'  tradingClass   : {opt.tradingClass}')
    print(f'  multiplier     : {opt.multiplier}')
    print(f'  currency       : {opt.currency}')

=== Qualifizierter Options-Kontrakt ===
  conId          : 814191620
  symbol         : AAPL
  expiry         : 20280121
  strike         : 250.0
  right          : P
  exchange       : SMART
  tradingClass   : AAPL
  multiplier     : 100
  currency       : USD


In [22]:
# ── Market-Data-Snapshot (Bid / Ask / Last / Greeks) ─────────────────────────
# genericTickList '106' = implizite Vola + modelGreeks
if not opt.conId:
    print('Übersprungen — Kontrakt nicht qualifiziert.')
else:
    t = ib.reqMktData(opt, '106', snapshot=False, regulatorySnapshot=False)
    ib.sleep(3.0)

    print('=== Option Market Data ===')
    print(f'  bid        : {t.bid}')
    print(f'  ask        : {t.ask}')
    print(f'  last       : {t.last}')
    print(f'  close      : {t.close}')
    print(f'  volume     : {t.volume}')

    # Berechnetes Mid
    def _num(x):
        try: f = float(x); return f if not math.isnan(f) else None
        except: return None

    bid, ask = _num(t.bid), _num(t.ask)
    if bid and ask and ask >= bid:
        mid = (bid + ask) / 2
        spread_pct = (ask - bid) / mid * 100 if mid else None
        print(f'\n  mid        : {mid:.4f}')
        print(f'  spread     : {ask - bid:.4f}  ({spread_pct:.1f}%)')
    else:
        print('\n  mid        : n/a (kein Bid/Ask)')

    # Greeks
    greeks = t.modelGreeks or t.lastGreeks or t.bidGreeks or t.askGreeks
    print()
    print('=== Greeks ===')
    if greeks:
        print(f'  impliedVol   : {greeks.impliedVol}')
        print(f'  delta        : {greeks.delta}')
        print(f'  gamma        : {greeks.gamma}')
        print(f'  theta        : {greeks.theta}')
        print(f'  vega         : {greeks.vega}')
        print(f'  undPrice     : {greeks.undPrice}')
    else:
        print('  Keine Greeks erhalten (Delayed-Data liefert oft keine Greeks).')

    ib.cancelMktData(opt)
    print('\n→ Stream abbestellt.')

=== Option Market Data ===
  bid        : 24.35
  ask        : 25.1
  last       : 24.55
  close      : 25.48
  volume     : 6.0

  mid        : 24.7250
  spread     : 0.7500  (3.0%)

=== Greeks ===
  impliedVol   : 0.3018517353935777
  delta        : -0.2984374508683406
  gamma        : 0.003418441494807312
  theta        : -0.02101067872115081
  vega         : 1.2442506697710378
  undPrice     : 273.47601318359375

→ Stream abbestellt.


---
## Disconnect

In [23]:
ib.disconnect()
print('Disconnected:', not ib.isConnected())

Disconnected: True
